In [1]:
import os, sys, glob, json, time, math
import numpy as np, pandas as pd
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/MSc Dissertation")
except Exception:
    BASE = Path(".")
print("BASE =", BASE.resolve())

B          = 2000      # bootstrap replicates
BLOCK_DAYS = 14        # block length; must exceed the duration of one storm
SEED       = 0
CI         = (2.5, 97.5)
rng = np.random.default_rng(SEED)
print(f"{B} replicates | {BLOCK_DAYS}-day blocks | {CI[0]}-{CI[1]} percentile intervals")

Mounted at /content/drive
BASE = /content/drive/MyDrive/MSc Dissertation
2000 replicates | 14-day blocks | 2.5-97.5 percentile intervals


In [2]:
SEARCH = [BASE, BASE / "ciew_v5_cache", Path(".")]
files = {}
for d in SEARCH:
    for f in sorted(glob.glob(str(d / "bootstrap_*.npz"))):
        files.setdefault(Path(f).name, f)
assert files, f"no bootstrap_*.npz found in {[str(s) for s in SEARCH]}"

DATA = {}
for name, path in files.items():
    z = np.load(path, allow_pickle=False)
    tag = name.replace("bootstrap_", "").replace(".npz", "")
    DATA[tag] = dict(P=z["P"], names=[str(n) for n in z["names"]],
                     y=z["y"].astype(np.int8), t=z["t"].astype(np.int64), path=path)
    print(f"{tag:<14} {z['P'].shape[0]} series x {z['y'].size:,} rows   {path}")
    for n in DATA[tag]["names"]:
        print(f"                 - {n}")

CIGB_v3        7 series x 2,615,600 rows   /content/drive/MyDrive/MSc Dissertation/bootstrap_CIGB_v3.npz
                 - Climatology
                 - Persistence
                 - Logistic Reg
                 - CIGB (ours)
                 - Croston
                 - SBA
                 - TSB
HEICH_CR_v2    6 series x 2,611,440 rows   /content/drive/MyDrive/MSc Dissertation/bootstrap_HEICH_CR_v2.npz
                 - Climatology
                 - Persistence
                 - HEICH-CR @24h
                 - Croston
                 - SBA
                 - TSB
HEICH_v3       8 series x 2,615,600 rows   /content/drive/MyDrive/MSc Dissertation/bootstrap_HEICH_v3.npz
                 - Climatology
                 - Persistence
                 - Linear only
                 - AI only
                 - Hybrid (calibrated)
                 - Croston
                 - SBA
                 - TSB
CIEW_v5b       8 series x 2,616,120 rows   /content/drive/MyDrive/MSc Dissertation

In [3]:
# Provenance: list every export with its modification time, series and row count
# so that a stale file left by a failed export in an earlier run is easy to spot.
import datetime as _dt

print(f"{'file':<34}{'modified':<22}{'series':>7}{'rows':>12}{'origins':>10}")
print("-" * 86)
_mtimes = {}
for tag, d in sorted(DATA.items()):
    _m = _dt.datetime.fromtimestamp(os.path.getmtime(d["path"]))
    _mtimes[tag] = _m
    print(f"{os.path.basename(d['path']):<34}{_m:%Y-%m-%d %H:%M}     "
          f"{d['P'].shape[0]:>5}{d['y'].size:>12,}{np.unique(d['t']).size:>10,}")

print()
_newest = max(_mtimes.values())
for tag, _m in sorted(_mtimes.items()):
    _age_days = (_newest - _m).days
    if _age_days > 1:
        print(f"  NOTE  {tag} was exported {_age_days} days before the newest file.")
print("  (Only the survival model was retrained; the other three exports are")
print("   expected to be older. Check that bootstrap_HEICH_CR_v2.npz is the")
print("   NEWEST of the four before trusting any interval involving it.)")

_surv = [t for t in DATA if "HEICH_CR" in t]
if _surv and _mtimes[_surv[0]] < _newest:
    print()
    print("  !! STOP: the survival export is not the newest file. Re-run cell 45")
    print("     of Survival_Wind_v3.ipynb before continuing, or every interval")
    print("     involving the survival model will be computed from old predictions.")

file                              modified               series        rows   origins
--------------------------------------------------------------------------------------
bootstrap_CIEW_v5b.npz            2026-09-10 18:13         8   2,616,120     5,031
bootstrap_CIGB_v3.npz             2026-09-04 07:07         7   2,615,600     5,030
bootstrap_HEICH_CR_v2.npz         2026-09-19 18:45         6   2,611,440     5,022
bootstrap_HEICH_v3.npz            2026-09-10 15:48         8   2,615,600     5,030

  NOTE  CIEW_v5b was exported 9 days before the newest file.
  NOTE  CIGB_v3 was exported 15 days before the newest file.
  NOTE  HEICH_v3 was exported 9 days before the newest file.
  (Only the survival model was retrained; the other three exports are
   expected to be older. Check that bootstrap_HEICH_CR_v2.npz is the
   NEWEST of the four before trusting any interval involving it.)


In [4]:
def reconstruct_keys(t):
    n = t.size
    uniq = np.unique(t)
    n_org = uniq.size
    assert n % n_org == 0, f"{n} rows is not a multiple of {n_org} origins"
    n_cell = n // n_org
    if t[0] == t[1]:                      # origin-major
        layout = "origin-major"
        cell = np.arange(n, dtype=np.int32) % n_cell
    else:                                 # cell-major
        layout = "cell-major"
        cell = (np.arange(n, dtype=np.int64) // n_org).astype(np.int32)
    return layout, n_org, n_cell, cell

for tag, d in DATA.items():
    layout, n_org, n_cell, cell = reconstruct_keys(d["t"])
    key = d["t"].astype(np.int64) * 100000 + cell          # unique composite key
    assert np.unique(key).size == key.size, f"{tag}: (origin, cell) pairs are not unique"
    d.update(cell=cell, key=key, layout=layout, n_org=n_org, n_cell=n_cell)
    print(f"{tag:<14} {layout:<12} {n_org:,} origins x {n_cell} cells   "
          f"base rate {d['y'].mean()*100:.2f}%   keys unique OK")

CIGB_v3        cell-major   5,030 origins x 520 cells   base rate 7.70%   keys unique OK
HEICH_CR_v2    origin-major 5,022 origins x 520 cells   base rate 7.71%   keys unique OK
HEICH_v3       origin-major 5,030 origins x 520 cells   base rate 7.70%   keys unique OK
CIEW_v5b       origin-major 5,031 origins x 520 cells   base rate 7.68%   keys unique OK


In [5]:
def prep(scores, y, block):
    # Sort once. Every replicate is then just a weight vector over this fixed order.
    order = np.argsort(-scores.astype(np.float64), kind="mergesort")
    s = scores[order]
    grp_end = np.r_[np.flatnonzero(np.diff(s)), s.size - 1]   # last row of each tied group
    ys = y[order].astype(np.float64)
    ps = scores[order].astype(np.float64)
    return dict(order=order, y=ys, blk=block[order], grp_end=grp_end, p=ps,
                sq=(ps - ys) ** 2)          # precomputed so Brier is a single dot product

def metrics(pre, w, ap_only=False):
    # Weighted PR-AUC, ROC-AUC and Brier from the pre-sorted arrays.
    # Inside the replicate loop only PR-AUC is needed, so ap_only skips the rest.
    ge = pre["grp_end"]
    tp = np.cumsum(w * pre["y"])[ge]
    fp = np.cumsum(w)[ge] - tp
    P, N = tp[-1], fp[-1]
    if P <= 0 or N <= 0:
        return np.nan, np.nan, np.nan
    prec = tp / np.maximum(tp + fp, 1e-12)
    rec  = tp / P
    ap   = float(np.sum(np.diff(np.r_[0.0, rec]) * prec))
    if ap_only:
        return ap, np.nan, np.nan
    auc  = float(np.trapezoid(np.r_[0.0, tp / P], np.r_[0.0, fp / N])
                 if hasattr(np, "trapezoid") else
                 np.trapz(np.r_[0.0, tp / P], np.r_[0.0, fp / N]))
    brier = float(np.dot(w, pre["sq"]) / np.sum(w))
    return ap, auc, brier

In [6]:
NBIN = 50000   # score bins; each replicate becomes a small matrix product

def block_ids(t, block_days=BLOCK_DAYS):
    return ((t - t.min()) // (24 * block_days)).astype(np.int64)

def prep_binned(scores, y, blk_idx, nblk, nbin=NBIN):
    # Sort by score once, group the sorted order into bins, then count positives
    # and negatives per (bin, block). A replicate is then Mpos @ w and Mneg @ w
    # over a few tens of thousands of bins rather than a cumulative sum over 2.6
    # million rows.
    # Bins are unions of ADJACENT TIE GROUPS, never splits of one. Splitting a tie
    # would let the metric reward an ordering that the model never expressed: a
    # constant series such as climatology would score as though it discriminated.
    # Snapping bin edges to score changes keeps a constant series exact.
    n = scores.size
    ss = scores.astype(np.float64)
    order = np.argsort(-ss, kind="mergesort")
    srt = ss[order]
    ys = y[order].astype(np.float64)
    b = blk_idx[order]
    grp = np.r_[0, np.cumsum(np.diff(srt) != 0)]
    ngrp = int(grp[-1]) + 1
    m = max(1, int(np.ceil(ngrp / int(nbin))))
    binid = (grp // m).astype(np.int64)
    nbin = int(binid[-1]) + 1
    flat = binid * nblk + b
    size = nbin * nblk
    Mpos = np.bincount(flat, weights=ys,       minlength=size).reshape(nbin, nblk)
    Mneg = np.bincount(flat, weights=1.0 - ys, minlength=size).reshape(nbin, nblk)
    return dict(Mpos=Mpos, Mneg=Mneg, order=order, y=ys,
                p=scores[order].astype(np.float64),
                sq=(scores[order].astype(np.float64) - ys) ** 2, blk=b)

def ap_binned(pre, w):
    tp = np.cumsum(pre["Mpos"] @ w)
    fp = np.cumsum(pre["Mneg"] @ w)
    P, N = tp[-1], fp[-1]
    if P <= 0 or N <= 0:
        return np.nan
    prec = tp / np.maximum(tp + fp, 1e-12)
    rec  = tp / P
    return float(np.sum(np.diff(np.r_[0.0, rec]) * prec))

def run_bootstrap(P, names, y, t, B=B, seed=SEED, label=""):
    # Returns exact point estimates and the replicate matrix for PR-AUC.
    blk = block_ids(t)
    ublk, blk_idx = np.unique(blk, return_inverse=True)
    nb = ublk.size
    ones = np.ones(y.size)

    exact, pres = {}, []
    for i, n in enumerate(names):
        exact[n] = metrics(prep(P[i], y, blk_idx), ones)[0]     # exact, unbinned
        pres.append(prep_binned(P[i], y, blk_idx, nb))
    err = max(abs(exact[n] - ap_binned(pres[i], np.ones(nb)))
              for i, n in enumerate(names))
    print(f"  {label}: {nb} blocks | binning error in PR AUC {err:.2e}", flush=True)

    r = np.random.default_rng(seed)
    reps = np.full((len(names), B), np.nan)
    t0 = time.time()
    for b in range(B):
        w = np.bincount(r.integers(0, nb, nb), minlength=nb).astype(np.float64)
        for i, pre in enumerate(pres):
            reps[i, b] = ap_binned(pre, w)
    print(f"  {label}: {B} replicates in {time.time()-t0:.0f}s", flush=True)
    return exact, reps

def summarise(point, reps, names):
    rows = []
    for i, n in enumerate(names):
        v = reps[i][np.isfinite(reps[i])]
        lo, hi = np.percentile(v, CI)
        rows.append({"Model": n, "PR AUC": point[n],
                     f"lo {CI[0]}%": lo, f"hi {CI[1]}%": hi, "width": hi - lo})
    return pd.DataFrame(rows).set_index("Model").sort_values("PR AUC", ascending=False)

In [7]:
RESULT, REPS = {}, {}
for tag, d in DATA.items():
    print(f"\n{tag}")
    point, reps = run_bootstrap(d["P"], d["names"], d["y"], d["t"], label=tag)
    RESULT[tag] = summarise(point, reps, d["names"])
    REPS[tag] = dict(reps=reps, names=d["names"])
    print(RESULT[tag].round(4).to_string())


CIGB_v3
  CIGB_v3: 45 blocks | binning error in PR AUC 4.34e-05
  CIGB_v3: 2000 replicates in 31s
              PR AUC  lo 2.5%  hi 97.5%   width
Model                                          
CIGB (ours)   0.5806   0.4971    0.6504  0.1533
Logistic Reg  0.5240   0.4436    0.5991  0.1555
Persistence   0.4759   0.4021    0.5417  0.1396
TSB           0.2786   0.2178    0.3287  0.1109
SBA           0.0859   0.0574    0.1168  0.0594
Croston       0.0855   0.0570    0.1166  0.0596
Climatology   0.0770   0.0532    0.1038  0.0506

HEICH_CR_v2
  HEICH_CR_v2: 45 blocks | binning error in PR AUC 3.42e-05
  HEICH_CR_v2: 2000 replicates in 30s
               PR AUC  lo 2.5%  hi 97.5%   width
Model                                           
HEICH-CR @24h  0.5701   0.4825    0.6482  0.1657
Persistence    0.4760   0.4021    0.5417  0.1397
TSB            0.2806   0.2194    0.3310  0.1117
SBA            0.0861   0.0575    0.1170  0.0595
Croston        0.0859   0.0571    0.1168  0.0597
Climatology    

In [8]:
def paired(reps, names, a, b):
    i, j = names.index(a), names.index(b)
    d = reps[i] - reps[j]
    d = d[np.isfinite(d)]
    lo, hi = np.percentile(d, CI)
    return dict(A=a, B=b, diff=float(np.mean(d)), lo=float(lo), hi=float(hi),
                p_A_better=float(np.mean(d > 0)),
                verdict="A better" if lo > 0 else ("B better" if hi < 0 else "not separable"))

for tag, R in REPS.items():
    names = R["names"]
    main = names[-4] if len(names) >= 4 else names[0]
    rows = [paired(R["reps"], names, names[0], o) for o in names[1:]] if False else []
    # compare every pair once
    rows = [paired(R["reps"], names, names[i], names[j])
            for i in range(len(names)) for j in range(i + 1, len(names))]
    df = pd.DataFrame(rows)
    print(f"\n{tag}: paired differences in PR AUC")
    print(df.round(4).to_string(index=False))


CIGB_v3: paired differences in PR AUC
           A            B    diff      lo      hi  p_A_better       verdict
 Climatology  Persistence -0.3966 -0.4523 -0.3376      0.0000      B better
 Climatology Logistic Reg -0.4451 -0.5104 -0.3778      0.0000      B better
 Climatology  CIGB (ours) -0.5005 -0.5640 -0.4313      0.0000      B better
 Climatology      Croston -0.0085 -0.0200  0.0017      0.0515 not separable
 Climatology          SBA -0.0089 -0.0205  0.0014      0.0445 not separable
 Climatology          TSB -0.1994 -0.2421 -0.1541      0.0000      B better
 Persistence Logistic Reg -0.0485 -0.0765 -0.0232      0.0000      B better
 Persistence  CIGB (ours) -0.1040 -0.1286 -0.0793      0.0000      B better
 Persistence      Croston  0.3881  0.3306  0.4454      1.0000      A better
 Persistence          SBA  0.3876  0.3304  0.4444      1.0000      A better
 Persistence          TSB  0.1972  0.1675  0.2272      1.0000      A better
Logistic Reg  CIGB (ours) -0.0555 -0.0745 -0.0357

In [9]:
MAIN = {   # tag -> the series in that file to treat as its headline model
 "CIEW_v5b":    "CIEW v4 (MAE-pretrained) calibrated",
 "CIGB_v3":     "CIGB (ours)",
 "HEICH_v3":    "Hybrid (calibrated)",
 "HEICH_CR_v2": "HEICH-CR @24h",
}
BASELINE_WORDS = ("climatology", "persistence", "croston", "sba", "tsb",
                  "logistic", "linear only", "ai only")

def pick(names, want):
    if want and want in names:
        return want
    cand = [n for n in names if not any(b in n.lower() for b in BASELINE_WORDS)]
    if not cand:
        cand = list(names)
    cal = [n for n in cand if "calib" in n.lower()]
    choice = (cal or cand)[0]
    print(f"    (exact name {want!r} not found, falling back to {choice!r})")
    return choice

print("Origin grids, which is why a row-level join is not possible:")
for tag, d in DATA.items():
    o = np.unique(d["t"])
    print(f"  {tag:<14} {o.size:,} origins | first hour {o.min()} "
          f"| hour mod 3 = {sorted(set((o % 3).tolist()))}")

# One block index shared by every file, anchored on the earliest origin overall.
T0 = min(int(d["t"].min()) for d in DATA.values())
NBLK = max(int((d["t"].max() - T0) // (24 * BLOCK_DAYS)) for d in DATA.values()) + 1
print()
print(f"common grid: {NBLK} blocks of {BLOCK_DAYS} days")

sets, names4 = {}, []
for tag, d in DATA.items():
    nm = pick(d["names"], MAIN.get(tag, ""))
    gb = ((d["t"] - T0) // (24 * BLOCK_DAYS)).astype(np.int64)
    sets[tag] = prep_binned(d["P"][d["names"].index(nm)], d["y"], gb, NBLK)
    names4.append(f"{tag} ({nm})")
    print(f"  {tag:<14} {nm}")

Origin grids, which is why a row-level join is not possible:
  CIGB_v3        5,030 origins | first hour 480122 | hour mod 3 = [2]
  HEICH_CR_v2    5,022 origins | first hour 480122 | hour mod 3 = [2]
  HEICH_v3       5,030 origins | first hour 480122 | hour mod 3 = [2]
  CIEW_v5b       5,031 origins | first hour 480120 | hour mod 3 = [0]

common grid: 45 blocks of 14 days
  CIGB_v3        CIGB (ours)
  HEICH_CR_v2    HEICH-CR @24h
  HEICH_v3       Hybrid (calibrated)
  CIEW_v5b       CIEW v4 (MAE-pretrained) calibrated


In [10]:
ones = np.ones(NBLK)
point4 = {n: ap_binned(sets[t], ones) for n, t in zip(names4, sets)}

r = np.random.default_rng(SEED)
reps4 = np.full((len(sets), B), np.nan)
t0 = time.time()
for b in range(B):
    w = np.bincount(r.integers(0, NBLK, NBLK), minlength=NBLK).astype(np.float64)
    for i, t in enumerate(sets):
        reps4[i, b] = ap_binned(sets[t], w)
print(f"{B} replicates over {NBLK} shared blocks in {time.time()-t0:.0f}s")
print()
print(summarise(point4, reps4, names4).round(4).to_string())

2000 replicates over 45 shared blocks in 11s

                                                PR AUC  lo 2.5%  hi 97.5%   width
Model                                                                            
CIEW_v5b (CIEW v4 (MAE-pretrained) calibrated)  0.6232   0.5430    0.6878  0.1448
CIGB_v3 (CIGB (ours))                           0.5806   0.4971    0.6504  0.1533
HEICH_CR_v2 (HEICH-CR @24h)                     0.5701   0.4825    0.6482  0.1657
HEICH_v3 (Hybrid (calibrated))                  0.5427   0.4592    0.6155  0.1563


In [11]:
rows = [paired(reps4, names4, names4[i], names4[j])
        for i in range(len(names4)) for j in range(i + 1, len(names4))]
CROSS = pd.DataFrame(rows).sort_values("diff", ascending=False)
print("Paired differences in PR AUC, blocks shared, rows each model's own")
print(CROSS.round(4).to_string(index=False))

pd.concat({t: R for t, R in RESULT.items()}).to_csv(BASE / "bootstrap_results.csv")
CROSS.to_csv(BASE / "bootstrap_paired_differences.csv", index=False)
print()
print(f"written -> {BASE / 'bootstrap_results.csv'}")
print(f"written -> {BASE / 'bootstrap_paired_differences.csv'}")

Paired differences in PR AUC, blocks shared, rows each model's own
                             A                                              B    diff      lo      hi  p_A_better       verdict
         CIGB_v3 (CIGB (ours))                 HEICH_v3 (Hybrid (calibrated))  0.0377  0.0265  0.0486      1.0000      A better
   HEICH_CR_v2 (HEICH-CR @24h)                 HEICH_v3 (Hybrid (calibrated))  0.0270  0.0139  0.0377      1.0000      A better
         CIGB_v3 (CIGB (ours))                    HEICH_CR_v2 (HEICH-CR @24h)  0.0107 -0.0029  0.0253      0.9365 not separable
         CIGB_v3 (CIGB (ours)) CIEW_v5b (CIEW v4 (MAE-pretrained) calibrated) -0.0428 -0.0665 -0.0186      0.0000      B better
   HEICH_CR_v2 (HEICH-CR @24h) CIEW_v5b (CIEW v4 (MAE-pretrained) calibrated) -0.0535 -0.0820 -0.0235      0.0005      B better
HEICH_v3 (Hybrid (calibrated)) CIEW_v5b (CIEW v4 (MAE-pretrained) calibrated) -0.0805 -0.1056 -0.0521      0.0000      B better

written -> /content/drive/MyDrive/MS